# FairDream — transparent detection and correction by reweighting on Census

This notebook demonstrates the full **FairDream** workflow on the Adult/Census binary classification task:

1. train a statistically performant baseline classifier;
2. detect whether the baseline creates unfair gaps between groups;
3. make the correction mechanism transparent by computing the group weights used during retraining;
4. train several reweighted fair candidates;
5. select the best model according to the chosen statistical/fairness trade-off.

The example focuses on **transparent in-processing reweighting**: the model is not post-processed after training and the true labels are not rewritten. Instead, the training loss gives more weight to errors made on previously disadvantaged groups. This is the mechanism described in `fairdream/correction.py` and discussed in the paper [Fairness Interventions: A Study in AI Explainability](https://arxiv.org/abs/2407.14766).

## 0. What “fairness correction” means here

In this notebook, the user chooses:

- a **protected attribute**, here `sex`;
- a **fairness purpose**, here `overall_positive_rate`, which corresponds to a Demographic-Parity-style objective: reduce the gap in positive predictions between groups;
- a **statistical criterion**, here `auc`, to avoid selecting a fair-looking model that performs poorly;
- a **trade-off**, here `fair_preferred`, meaning that the final model selection gives more importance to the fairness score than to the statistical score.

FairDream then creates several new candidate classifiers. Each candidate receives a different set of **sample weights**. A group with a lower baseline fairness score receives a higher error weight, so the next classifier is trained to “pay more attention” to mistakes made on that group.

This is intentionally transparent: the group gaps, the number of impacted individuals, and the resulting weights can be inspected before the corrected model is trained.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys
sys.path.append('../')

import math
import numpy as np
import pandas as pd
from matplotlib import pyplot

from fairdream.data_preparation import *
from fairdream.compute_scores import *
from fairdream.detection import *
from fairdream.correction import *
from fairdream.plots import *

## 1. Configure the experiment

We keep the task deliberately simple: binary classification on tabular data.

The notebook name mentions XGBoost, so the default model is `xgboost`. If you want a faster local run, you can temporarily switch to `random_forest` or `log_reg`, both supported by the correction code.

In [ ]:
# Statistical task
model_task = 'classification'
stat_criteria = 'auc'

# Baseline learner used before and after fairness correction.
# Other supported options include: 'random_forest', 'log_reg', 'mlp', 'svm'.
model_type = 'xgboost'

# Fairness objective
protected_attribute = 'sex'
fairness_purpose = 'overall_positive_rate'

# FairDream model-selection preference
tradeoff = 'fair_preferred'  # {'moderate', 'fair_preferred', 'stat_preferred'}
weight_method = 'weighted_groups'
nb_fair_models = 4

## 2. Load the Adult/Census binary classification dataset

The target is whether an individual earns more than `$50K`. The model predicts a positive label for `>50K`.

We keep the protected attribute inside `X` because FairDream needs it both for detection and for the reweighting step. In production, you should decide carefully whether the protected attribute is allowed in training, monitoring, both, or neither, depending on the legal and business context.

In [ ]:
from sklearn.datasets import fetch_openml

data = fetch_openml(data_id=1590, as_frame=True)

X = data.data.copy()
Y = (data.target == '>50K').astype(int)

dataset = X.copy()
dataset['target'] = Y

print(f'X shape: {X.shape}')
print(f'Positive target rate: {Y.mean():.3f}')
dataset.head()

### 2.1. Before any model: inspect base rates by group

A fairness intervention should not start from a black box. First, inspect the empirical distribution of the true labels by group.

This table is not yet a fairness diagnosis. It only says how the observed target is distributed in the dataset. The model may amplify, reduce, or transform these disparities.

In [ ]:
base_rates = (
    dataset
    .groupby(protected_attribute)
    .agg(
        nb_individuals=('target', 'size'),
        nb_positive=('target', 'sum'),
        target_positive_rate=('target', 'mean'),
    )
    .sort_values('target_positive_rate')
)

base_rates

## 3. Train an unconstrained baseline model

This first model optimizes the statistical criterion only. It is the reference model whose behavior will be inspected.

FairDream’s correction is easier to explain if this distinction is explicit:

- **baseline model**: trained without any fairness-oriented reweighting;
- **corrected candidates**: trained with group-specific sample weights;
- **selected corrected model**: the candidate with the best stat/fair trade-off.

In [ ]:
X_train, X_valid, X_train_valid, X_test, Y_train, Y_valid, Y_train_valid, Y_test = train_valid_test_split(
    X,
    Y,
    model_task,
)

save_model = True
uncorrected_model_path = '/work/data/models/uncorrected_model.pkl'

Y_pred_train_valid = train_naive_model(
    X_train,
    X_valid,
    X_train_valid,
    X_test,
    Y_train,
    Y_valid,
    Y_train_valid,
    Y_test,
    model_task,
    stat_criteria,
    save_model=save_model,
    model_type=model_type,
)

In [ ]:
train_valid_set_with_uncorrected_results = augment_train_valid_set_with_results(
    'uncorrected',
    X_train_valid,
    Y_train_valid,
    Y_pred_train_valid,
    model_task,
)

train_valid_set_with_uncorrected_results.head()

## 4. Detection: does the baseline treat groups differently?

Detection is a descriptive step. It asks whether the trained model has learned a behavior that creates large gaps between groups.

Here, `overall_positive_rate` asks: among individuals in each group, what share receives a positive prediction from the model?

The alert threshold is user-defined:

- `injustice_acceptance = 3` means that a group ratio larger than 3:1 is considered large enough to alert the user;
- `min_individuals_discrimined = 0.01` avoids reporting tiny groups that are too small to interpret reliably.

In [ ]:
augmented_train_valid_set = train_valid_set_with_uncorrected_results
model_name = 'uncorrected'

injustice_acceptance = 3
min_individuals_discrimined = 0.01

discrimination_alert(
    augmented_train_valid_set,
    model_name,
    fairness_purpose,
    model_task,
    injustice_acceptance,
    min_individuals_discrimined,
)

### 4.1. Turn the alert into a group-level score table

The correction step reuses the same fairness purpose. Before training corrected candidates, we compute the baseline group scores explicitly.

For `overall_positive_rate`, a **lower** value identifies a group that is selected less often by the baseline model. This is the group that the reweighting mechanism will prioritize.

In [ ]:
fair_scores_df_uncorrected = fair_score(
    train_valid_set_with_uncorrected_results,
    'uncorrected',
    fairness_purpose,
    model_task,
    protected_attribute,
    fairness_mode='correction',
)

fair_scores_df_uncorrected.sort_values(fairness_purpose)

## 5. Correction principle: transparent group reweighting

FairDream’s `weighted_groups` correction does **not** directly force the model to output the same positive rate for every group. It changes the training loss through sample weights.

For the current fairness purpose, the implementation in `fairdream/correction.py` follows this intuition:

1. compute the baseline fairness score of each group;
2. identify the best-treated group for the selected fairness purpose;
3. compute each group’s disadvantage gap;
4. multiply the gap by the group size, because a large affected group should matter more than a tiny one;
5. convert this into a candidate-specific sample weight;
6. train several models with different weights;
7. select the candidate with the best trade-off between statistical and fairness scores.

The important explainability point is that the correction remains attached to the usual prediction loss. If a disadvantaged individual is truly negative, giving that individual a larger weight also makes a false positive more costly. This is why FairDream may reduce gaps while tending toward true-label-conditioned criteria such as Equalized Odds, rather than mechanically enforcing Demographic Parity.

### 5.1. Reproduce the sample-weight calculation before training

The next helper mirrors the current `weighted_groups_fair_train` logic for the `overall_positive_rate` case. Its purpose is pedagogical: it shows the numbers that will later be hidden inside `fair_train`.

Read the resulting table as follows:

- `group_score`: the baseline group score for `overall_positive_rate`;
- `score_gap_to_best`: how far the group is from the best-treated group;
- `estimated_impacted_individuals`: score gap multiplied by group size;
- `relative_impacted_share`: impacted individuals normalized by total training-validation population;
- `candidate_weight`: sample weight assigned to individuals from that group for a given corrected candidate.

In [ ]:
def explain_fairdream_reweighting(
        fair_scores_df: pd.DataFrame,
        fairness_purpose: str,
        nb_fair_models: int,
    ) -> pd.DataFrame:
        """Pedagogical mirror of the weighted_groups logic used in correction.py.

        This helper is intentionally kept in the notebook so users can inspect the
        mechanism before calling fair_train(). It is written for the current
        single-attribute correction example.
        """
        total_individuals = fair_scores_df['nb_individuals_by_group'].sum()
        max_fair_score = fair_scores_df[fairness_purpose].max()
        min_fair_score = fair_scores_df[fairness_purpose].min()
        maximise = is_fairness_purpose_to_maximise(fairness_purpose)

        rows = []
        for fair_model_number in range(nb_fair_models):
            for group_name, group_row in fair_scores_df.iterrows():
                group_score = group_row[fairness_purpose]

                if maximise:
                    best_fair_score = max_fair_score
                    score_gap_to_best = best_fair_score - group_score
                else:
                    best_fair_score = min_fair_score
                    score_gap_to_best = group_score - best_fair_score

                estimated_impacted_individuals = (
                    score_gap_to_best * group_row['nb_individuals_by_group']
                )
                relative_impacted_share = estimated_impacted_individuals / total_individuals

                # Same candidate variety used in correction.py: even-numbered models
                # keep coefficient 1; odd-numbered models stress the relative impact.
                coeff_disadvantaged = (
                    fair_model_number * relative_impacted_share
                    if fair_model_number % 2 != 0
                    else 1
                )

                candidate_weight = coeff_disadvantaged * math.exp(
                    fair_model_number * (max_fair_score - group_score)
                )

                rows.append(
                    {
                        'candidate_model': f'weighted_fair_{fair_model_number}',
                        'group': group_name,
                        'nb_individuals_by_group': group_row['nb_individuals_by_group'],
                        'group_score': group_score,
                        'score_gap_to_best': score_gap_to_best,
                        'estimated_impacted_individuals': estimated_impacted_individuals,
                        'relative_impacted_share': relative_impacted_share,
                        'candidate_weight': candidate_weight,
                    }
                )

        return pd.DataFrame(rows)


weights_explanation = explain_fairdream_reweighting(
    fair_scores_df_uncorrected,
    fairness_purpose,
    nb_fair_models,
)

weights_explanation.sort_values(['candidate_model', 'candidate_weight'], ascending=[True, False])

### 5.2. Show how weights attach to individuals

The group score is copied onto each individual in the training-validation set. The score is then mapped to the candidate weight.

This is the concrete “individual reweighting” step: the model still receives rows representing individuals, but their training errors no longer have the same importance.

In [ ]:
candidate_to_inspect = 'weighted_fair_1'

score_to_weight = (
    weights_explanation
    .loc[weights_explanation['candidate_model'] == candidate_to_inspect]
    .set_index('group_score')['candidate_weight']
    .to_dict()
)

individual_weight_preview = train_valid_set_with_uncorrected_results[[
    protected_attribute,
    'target_train_valid',
    'predicted_uncorrected',
    f'{fairness_purpose}_uncorrected',
]].copy()

individual_weight_preview['sample_weight_for_candidate'] = (
    individual_weight_preview[f'{fairness_purpose}_uncorrected'].map(score_to_weight)
)

individual_weight_preview.head(10)

## 6. Train corrected candidates with FairDream

Now we call `fair_train`, which performs the complete correction workflow:

- compute the baseline stat/fair scores;
- train `nb_fair_models` reweighted candidates;
- score each candidate;
- select the best candidate according to `tradeoff`.

The candidate model names are `weighted_fair_0`, `weighted_fair_1`, and so on. The baseline remains named `uncorrected`.

In [ ]:
train_valid_set_with_corrected_results, models_df, best_model_dict = fair_train(
    X=X,
    Y=Y,
    train_valid_set_with_uncorrected_results=train_valid_set_with_uncorrected_results,
    protected_attribute=protected_attribute,
    fairness_purpose=fairness_purpose,
    model_task=model_task,
    stat_criteria=stat_criteria,
    tradeoff=tradeoff,
    weight_method=weight_method,
    nb_fair_models=nb_fair_models,
    model_type=model_type,
)

best_model_dict['model_name']

## 7. Inspect the model-selection table

FairDream does not assume that the strongest reweighting is always the best answer. It trains several candidates and compares them.

The selected model is the one maximizing the trade-off score:

- `stat_score_value`: predictive performance score;
- `fair_score_value`: aggregated fairness score across groups;
- `tradeoff_score`: combined score according to `tradeoff`.

In [ ]:
top_models = models_df.sort_values(by='tradeoff_score', ascending=False)
top_models[[
    'model_name',
    'fair_score_value',
    'stat_score_value',
    'tradeoff_score',
    'selected',
]] if 'selected' in top_models.columns else top_models

In [ ]:
fair_model_results(
    train_valid_set_with_corrected_results,
    models_df,
    best_model_dict,
    protected_attribute,
    fairness_purpose,
    model_task,
)

## 8. Compare the baseline and the selected corrected model

The next plot compares several classification fairness metrics before and after correction:

- `overall_positive_rate`: unconditional selection rate;
- `true_positive_rate` and `false_positive_rate`: Equalized-Odds components;
- `true_negative_rate` and `false_negative_rate`: complementary error profile.

This is important because a correction requested under one fairness purpose may have its most visible effects on another fairness metric. In the FairDream paper, this is central to explaining why transparent reweighting can tend toward Equalized Odds even when the user starts from a Demographic-Parity-style objective.

In [ ]:
list_compared_metrics = [
    'nb_positive',
    'overall_positive_rate',
    'true_positive_rate',
    'true_negative_rate',
    'false_positive_rate',
    'false_negative_rate',
]

plot_compared_metrics(
    train_valid_set_with_corrected_results=train_valid_set_with_corrected_results,
    protected_attribute=protected_attribute,
    fairness_purpose=fairness_purpose,
    model_task=model_task,
    best_model_dict=best_model_dict,
    list_compared_metrics=list_compared_metrics,
)

### 8.1. Sanity check: labels were not rewritten

Reweighting changes the cost of training errors. It should not change the observed target labels.

In [ ]:
unchanged_labels = (
    train_valid_set_with_corrected_results['target_train_valid'].reset_index(drop=True)
    == train_valid_set_with_uncorrected_results['target_train_valid'].reset_index(drop=True)
).all()

print(f'Target labels unchanged by correction: {unchanged_labels}')

## 9. How to explain the correction to a non-technical stakeholder

A compact explanation is:

> The first model selected one group less often. FairDream measured this gap, converted it into group-specific training weights, and retrained several candidate models. Errors on the previously disadvantaged group became more expensive during training. The final model was selected because it improved the fairness objective while preserving enough predictive performance.

Two caveats should always be stated:

1. Reweighting does not guarantee exact Demographic Parity. Because the weights are still applied inside the ordinary prediction loss, the model remains sensitive to the distribution of true labels.
2. A fairness objective is normative. The practitioner must justify why `overall_positive_rate`, `true_positive_rate`, `false_positive_rate`, or another metric is the relevant one for the use case.

## 10. Next experiments

To stress-test the correction, rerun the notebook with:

- another `protected_attribute`, such as `age`, `race`, or `native-country`;
- another fairness purpose, such as `false_negative_rate` if missed positives are the critical harm;
- another model type, such as `random_forest` or `log_reg`;
- another trade-off, especially `moderate` and `stat_preferred`.

The key question is not only “did the fairness score improve?”, but also “which fairness metric improved, which one did not, and why?”. That explanatory step is the core of the FairDream approach.